In [1]:
import nltk
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from gensim.models import Word2Vec
from torch.nn.utils.rnn import pad_sequence
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df = pd.read_csv("training.1600000.processed.noemoticon.csv", encoding='latin-1')

In [3]:
df.head()

,polarity of tweet,id of the tweet,date of the tweet,query,user,text of the tweet
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [4]:
df['polarity of tweet\xa0'].value_counts()

polarity of tweet 
0    799996
4    248576
Name: count, dtype: int64

In [6]:
df['text of the tweet\xa0'].head()

0    is upset that he can't update his Facebook by ...
1    @Kenichan I dived many times for the ball. Man...
2      my whole body feels itchy and like its on fire 
3    @nationwideclass no, it's not behaving at all....
4                        @Kwesidei not the whole crew 
Name: text of the tweet , dtype: object

Only 0 and 4 polarity in the data, no other polarity present

In [5]:
import re
import nltk

def preprocess_text(text):
    # Check if the input is a string; if not, return empty list
    if not isinstance(text, str):
        return []
    
    # Step 1: Remove URLs (http, https, www)
    text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
    
    # Step 2: Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Step 3: Remove special characters and punctuation, keep only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Step 4: Convert to lowercase
    text = text.lower()
    
    # Step 5: Remove extra whitespaces (replace multiple spaces with single space)
    text = re.sub(r'\s+', ' ', text)
    
    # Step 6: Strip leading/trailing spaces
    text = text.strip()
    
    # Step 7: Tokenize (if text is empty after cleaning, return empty list)
    if not text:
        return []
    
    tokens = nltk.word_tokenize(text)
    
    return tokens

In [7]:
df['text'] = df['text of the tweet\xa0'].apply(preprocess_text)

In [8]:
df.head()

,polarity of tweet,id of the tweet,date of the tweet,query,user,text of the tweet,text
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,"[is, upset, that, he, can, t, update, his, fac..."
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,"[kenichan, i, dived, many, times, for, the, ba..."
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,"[my, whole, body, feels, itchy, and, like, its..."
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....","[nationwideclass, no, it, s, not, behaving, at..."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew,"[kwesidei, not, the, whole, crew]"


In [9]:
df['text_len'] = df['text'].apply(lambda ls: len(ls))

In [11]:
df['text_len'].max()

52

In [12]:
# Fetch embeddings
word2vec_model = Word2Vec(sentences=df.text.values.tolist(), 
                          vector_size=100, min_count=1, workers=4)

# Get vocabulary size
vocab_size = len(word2vec_model.wv)
print(vocab_size)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


406721


In [13]:
# Convert text to Word2Vec embeddings
def text_to_embeddings(text, word2vec_model, seq_length):
    embeddings = []
    
    for i, word in enumerate(text):
        if word in word2vec_model.wv:
            if i == seq_length:
                break
            embeddings.append(word2vec_model.wv[word])
        else:
            continue
        
    # Padding the sequences
    if len(embeddings) < seq_length:
        zero_padding = [np.zeros(word2vec_model.vector_size) \
                        for _ in range(seq_length - len(embeddings))]

        embeddings = embeddings + zero_padding

    return embeddings[:seq_length]

In [14]:
df['polarity'] = df['polarity of tweet\xa0'].map({0: 0, 4: 1})

In [16]:
data = df[['text', 'polarity']]

In [18]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048572 entries, 0 to 1048571
Data columns (total 2 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   text      1048572 non-null  object
 1   polarity  1048572 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 16.0+ MB


In [21]:
train_val_df, test_df = train_test_split(
    data,
    test_size=0.20,          # 20% for test
    random_state=42,         # for reproducibility
    stratify=data['polarity']  # stratify on the target column
)

In [22]:
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.10,          # 10% of the 80% → 8% of total
    random_state=42,
    stratify=train_val_df['polarity']
)

In [24]:
print(f"Original shape: {data.shape}")
print(f"Train shape:    {train_df.shape} ({len(train_df)/len(data)*100:.1f}%)")
print(f"Val shape:      {val_df.shape} ({len(val_df)/len(data)*100:.1f}%)")
print(f"Test shape:     {test_df.shape} ({len(test_df)/len(data)*100:.1f}%)")

Original shape: (1048572, 2)
Train shape:    (754971, 2) (72.0%)
Val shape:      (83886, 2) (8.0%)
Test shape:     (209715, 2) (20.0%)


In [25]:
print("\nClass distribution:")
print("Original:\n", data['polarity'].value_counts(normalize=True).round(3))
print("Train:\n", train_df['polarity'].value_counts(normalize=True).round(3))
print("Val:\n", val_df['polarity'].value_counts(normalize=True).round(3))
print("Test:\n", test_df['polarity'].value_counts(normalize=True).round(3))


Class distribution:
Original:
 polarity
0    0.763
1    0.237
Name: proportion, dtype: float64
Train:
 polarity
0    0.763
1    0.237
Name: proportion, dtype: float64
Val:
 polarity
0    0.763
1    0.237
Name: proportion, dtype: float64
Test:
 polarity
0    0.763
1    0.237
Name: proportion, dtype: float64


In [28]:
def prepare_data(reviews, labels, word2vec_model):
    X = [text_to_embeddings(review, word2vec_model, 50) for review in reviews]
    X = [torch.tensor(embeddings, dtype=torch.float32) for embeddings in X]
    y = torch.tensor(labels.values, dtype=torch.long)
    return X, y

In [29]:
X_train, y_train = prepare_data(train_df.text, train_df.polarity,
                    word2vec_model)

X_val, y_val = prepare_data(val_df.text, val_df.polarity,
                    word2vec_model)

X_test, y_test = prepare_data(test_df.text, test_df.polarity,
                              word2vec_model)

In [31]:
X_train[0].shape

torch.Size([50, 100])

In [33]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)


In [36]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [38]:
# define hyperparameters
input_size = word2vec_model.vector_size
hidden_size = 128
output_size = 1
num_layers = 2
learning_rate = 0.001
num_epochs = 20
batch_size = 64

In [39]:
# Create DataLoader
train_data = TensorDataset(torch.stack(X_train), y_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

In [40]:
# Create DataLoader
val_data = TensorDataset(torch.stack(X_val), y_val)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)

In [41]:
# Create DataLoader
test_data = TensorDataset(torch.stack(X_test), y_test)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

In [51]:
import time
import seaborn as sns
import matplotlib.pyplot as plt

In [53]:
# Model class (modified to be bidirectional)
class SentimentRNN(nn.Module):
    def __init__(self, cell_type, input_size, hidden_size, num_layers, output_size):
        super(SentimentRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.cell_type = cell_type
        self.bidirectional = True
        self.num_directions = 2 if self.bidirectional else 1
        
        if cell_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, bidirectional=self.bidirectional)
        elif cell_type == 'LSTM':
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=self.bidirectional)
        elif cell_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=self.bidirectional)
        else:
            raise ValueError("Invalid cell_type")
        
        # Linear layer input size: hidden_size * num_directions
        self.fc = nn.Linear(hidden_size * self.num_directions, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * self.num_directions, x.size(0), self.hidden_size).to(x.device)
        if self.cell_type == 'LSTM':
            c0 = torch.zeros(self.num_layers * self.num_directions, x.size(0), self.hidden_size).to(x.device)
            out, _ = self.rnn(x, (h0, c0))
        else:
            out, _ = self.rnn(x, h0)
        
        # For bidirectional, out[:, -1, :] is concat of last forward and first backward
        out = out[:, -1, :]
        out = self.fc(out)
        return out

# Evaluation function (modified from previous)
def evaluate_metrics(model, loader, criterion, device, return_probs=True):
    model.eval()
    loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(inputs)
            loss += criterion(outputs, labels).item()
            probs = torch.sigmoid(outputs).squeeze()
            predicted = (probs > 0.5).float()
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.squeeze().cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    acc = 100 * (all_preds == all_labels).sum() / len(all_labels)
    precision = precision_score(all_labels, all_preds, average=None)
    recall = recall_score(all_labels, all_preds, average=None)
    f1 = f1_score(all_labels, all_preds, average=None)
    cm = confusion_matrix(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs) if return_probs else None
    
    return loss / len(loader), acc, precision, recall, f1, cm, auc

# Function to train and evaluate a model
def train_and_evaluate(cell_type, train_loader, val_loader, test_loader, device):
    model = SentimentRNN(cell_type, input_size, hidden_size, num_layers, output_size).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    best_val_loss = float('inf')
    best_model_path = f'best_{cell_type}_model.pth'
    
    # Training time
    start_time = time.time()
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        correct, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_losses.append(train_loss / len(train_loader))
        train_accs.append(100 * correct / total)
        
        # Validation
        val_loss, val_acc, *rest = evaluate_metrics(model, val_loader, criterion, device, return_probs=False)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        print(f"{cell_type} Epoch {epoch+1}: Train Loss {train_losses[-1]:.4f}, Val Loss {val_losses[-1]:.4f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved for {cell_type} at epoch {epoch+1}")
    
    training_time = time.time() - start_time
    
    # Memory usage (approximate peak GPU memory if using CUDA)
    if torch.cuda.is_available():
        memory_usage = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
        torch.cuda.reset_peak_memory_stats()
    else:
        memory_usage = 'N/A (CPU)'
    
    # Load best model for test evaluation
    model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best {cell_type} model for test evaluation")
    
    # Test metrics
    test_loss, test_acc, precision, recall, f1, cm, auc = evaluate_metrics(model, test_loader, criterion, device)
    
    return {
        'model': model,
        'cell_type': cell_type,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'test_acc': test_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cm': cm,
        'auc': auc,
        'training_time': training_time,
        'memory_usage': memory_usage
    }

In [ ]:
# Train all models
models_results = {}
for cell in ['RNN', 'LSTM', 'GRU']:
    results = train_and_evaluate(cell, train_loader, val_loader, test_loader, device)
    models_results[cell] = results

/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


RNN Epoch 1: Train Loss 0.5498, Val Loss 0.5497
Best model saved for RNN at epoch 1


/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


RNN Epoch 2: Train Loss 0.5502, Val Loss 0.5494
Best model saved for RNN at epoch 2


/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

RNN Epoch 16: Train Loss 0.5506, Val Loss 0.5524


/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


RNN Epoch 17: Train Loss 0.5506, Val Loss 0.5482


/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


RNN Epoch 18: Train Loss 0.5507, Val Loss 0.5524


In [ ]:
# Print Comparative Report
print("Comparative Report:")
for cell, res in models_results.items():
    print(f"\n{cell} Model:")
    print(f"Performance on Test Set:")
    print(f"  Accuracy: {res['test_acc']:.2f}%")
    print(f"  Precision (0/1): {res['precision'][0]:.4f} / {res['precision'][1]:.4f}")
    print(f"  Recall (0/1): {res['recall'][0]:.4f} / {res['recall'][1]:.4f}")
    print(f"  F1-Score (0/1): {res['f1'][0]:.4f} / {res['f1'][1]:.4f}")
    print(f"  ROC-AUC: {res['auc']:.4f}")
    print(f"Computational Requirements:")
    print(f"  Training Time: {res['training_time']:.2f} seconds")
    print(f"  Memory Usage: {res['memory_usage']:.2f} MB" if isinstance(res['memory_usage'], float) else res['memory_usage'])
    print(f"Complexity of Implementation:")
    print(f"  Parameters: {sum(p.numel() for p in res['model'].parameters() if p.requires_grad)}")
    print(f"  {cell} has {'standard' if cell == 'RNN' else 'more gates than RNN'}, LSTM is most complex with forget gate.")

# Visualizations
# 1. Bar chart for F1-scores
cells = list(models_results.keys())
f1_neg = [res['f1'][0] for res in models_results.values()]
f1_pos = [res['f1'][1] for res in models_results.values()]

x = np.arange(len(cells))
width = 0.35

fig, ax = plt.subplots()
ax.bar(x - width/2, f1_neg, width, label='Negative (0)')
ax.bar(x + width/2, f1_pos, width, label='Positive (1)')
ax.set_ylabel('F1-Score')
ax.set_title('F1-Scores Comparison')
ax.set_xticks(x)
ax.set_xticklabels(cells)
ax.legend()
plt.show()

# 2. Line plots for loss and accuracy over epochs
fig, axs = plt.subplots(2, 1, figsize=(10, 10))
for cell, res in models_results.items():
    axs[0].plot(range(1, num_epochs+1), res['train_losses'], label=f'{cell} Train Loss')
    axs[0].plot(range(1, num_epochs+1), res['val_losses'], linestyle='--', label=f'{cell} Val Loss')
    axs[1].plot(range(1, num_epochs+1), res['train_accs'], label=f'{cell} Train Acc')
    axs[1].plot(range(1, num_epochs+1), res['val_accs'], linestyle='--', label=f'{cell} Val Acc')

axs[0].set_title('Loss over Epochs')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[1].set_title('Accuracy over Epochs')
axs[1].set_ylabel('Accuracy (%)')
axs[1].set_xlabel('Epoch')
axs[1].legend()
plt.show()

# 3. Confusion matrix heatmaps
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for i, (cell, res) in enumerate(models_results.items()):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=axs[i])
    axs[i].set_title(f'{cell} Confusion Matrix')
    axs[i].set_xlabel('Predicted')
    axs[i].set_ylabel('True')

plt.tight_layout()
plt.show()

/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


RNN Epoch 1: Train Loss 0.5491, Val Loss 0.5494
Best model saved for RNN at epoch 1
RNN Epoch 2: Train Loss 0.5500, Val Loss 0.5611


/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


RNN Epoch 3: Train Loss 0.5506, Val Loss 0.5484
Best model saved for RNN at epoch 3


/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


RNN Epoch 20: Train Loss 0.5505, Val Loss 0.5533
Loaded best RNN model for test evaluation


/opt/miniconda3/envs/pg_python_ai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


LSTM Epoch 1: Train Loss 0.4015, Val Loss 0.3232
Best model saved for LSTM at epoch 1
LSTM Epoch 2: Train Loss 0.3098, Val Loss 0.3150
Best model saved for LSTM at epoch 2
LSTM Epoch 3: Train Loss 0.2940, Val Loss 0.3074
Best model saved for LSTM at epoch 3
LSTM Epoch 4: Train Loss 0.2813, Val Loss 0.3071
Best model saved for LSTM at epoch 4
LSTM Epoch 5: Train Loss 0.2690, Val Loss 0.3073
LSTM Epoch 6: Train Loss 0.2569, Val Loss 0.3094
LSTM Epoch 7: Train Loss 0.2463, Val Loss 0.3144
LSTM Epoch 8: Train Loss 0.2348, Val Loss 0.3274
LSTM Epoch 9: Train Loss 0.2237, Val Loss 0.3286
LSTM Epoch 10: Train Loss 0.2131, Val Loss 0.3387
LSTM Epoch 11: Train Loss 0.2039, Val Loss 0.3435
LSTM Epoch 12: Train Loss 0.1957, Val Loss 0.3548
LSTM Epoch 13: Train Loss 0.1882, Val Loss 0.3668
LSTM Epoch 14: Train Loss 0.1819, Val Loss 0.3697
LSTM Epoch 15: Train Loss 0.1768, Val Loss 0.3876
LSTM Epoch 16: Train Loss 0.1718, Val Loss 0.3849
LSTM Epoch 17: Train Loss 0.1674, Val Loss 0.4049
LSTM Epoch 

NameError: name 'plt' is not defined